# From Pixels to Predictions
**CSI 5550 - Visual Computing**

Implement sampling, quantization, affine rotation, histogram equalization,
Gaussian smoothing, and Laplacian sharpening.
Then measure how **your implementations** affect a trained image classifier.

## 1. Setup
We use ResNet-18, already trained on ImageNet, with weights supplied by
[TorchVision](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html).
No training is needed. We restrict its predictions to the ten Imagenette categories.
Each input image is a 224×224 RGB tensor with values in [0, 1].

In [ ]:
import math
import os
import sys
from importlib import reload
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import display
from torch.utils.data import DataLoader

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/cnn_signal_sensitivity").exists()
)
sys.path.insert(0, str(ROOT / "src"))
os.environ["TORCH_HOME"] = str(ROOT / "data/models")

from cnn_signal_sensitivity import checks, experiments
from cnn_signal_sensitivity.data import CachedInputs, case_studies, load_lab_data
from cnn_signal_sensitivity.evaluation import predict, run_suite, seed_everything
from cnn_signal_sensitivity.models import choose_device, load_classifier
from cnn_signal_sensitivity.plotting import (
    image_grid,
    plot_curves,
    transformation_gallery,
)
from cnn_signal_sensitivity.transforms import identity_condition

reload(checks)  # refresh checks if this kernel imported an older version
seed_everything(5550)
torch.set_num_threads(min(8, os.cpu_count() or 1))
DEVICE = choose_device()
OUTPUT = ROOT / "outputs/lab"
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Using {DEVICE}")

## 2. Original images and predictions
We use the same 200 images for every comparison. Accuracy is the percentage classified
correctly; confidence is the model's probability among the ten possible categories.

In [ ]:
dataset = CachedInputs(load_lab_data(ROOT / "data"))
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
model = load_classifier(DEVICE)
clean = predict(model, loader, identity_condition())
print(f"Original-image accuracy: {100 * clean.correct.mean():.1f}%")
case_images, case_labels, case_ids = case_studies(dataset)
fig = image_grid(case_images, case_labels)
plt.show()

Evaluated operations and parameter ranges:

| Operation | Image change | Tested settings |
|---|---|---|
| Sampling | Decimate & reconstruct | Factor 1,2,4,8,16 |
| Quantization | Fewer intensity levels | 8,6,4,3,2,1 bits |
| Affine rotation | Rotate, bilinear, gray fill | 0°, ±15°, ±30°, ±45° |
| Histogram equalization | Redistribute luminance | 256 bins (vs. gray) |
| Gaussian smoothing | Neighborhood blur | σ = 0,0.5,1,2,4 px |
| Laplacian sharpening | High-frequency boost | Amount 0,0.25,0.5,1,2 |

## 3. Implement the operators
Inputs are floating-point RGB tensors of shape (B,3,H,W), with values in [0,1].
Preserve shape, dtype, and device; do not overwrite the input in place.
Operators are independent: you can complete or test them in any order, or skip one and return later (D2 may call D1).

### A. Image sampling and interpolation
Sampling determines how finely an image's spatial detail is represented.
Keep every factor-th row and column. Reconstruct the original size using
nearest-neighbor interpolation: repeat retained samples factor times along height
and width, then crop to the original dimensions. Factor 1 reproduces the input.

In [ ]:
def sample_image(images: torch.Tensor, factor: int) -> torch.Tensor:
    """Reduce spatial sampling, then reconstruct using nearest neighbors.

    Args:
        images: Float RGB batch in [0, 1], shaped (B, 3, H, W).
            B is the number of images; H and W are height and width.
        factor: Positive integer spacing between retained rows and columns.

    Returns:
        Reconstructed batch with the same shape, dtype, and device as images.
        The input is not modified."""
    raise NotImplementedError("Implement sample_image to test this operator.")

In [ ]:
demo_input = case_images[1:2].clone()
factors = [2, 4, 8, 16]
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5), layout="constrained")
axes[0].imshow(demo_input[0].permute(1, 2, 0).cpu())
axes[0].set_title("Input image")
for ax, factor in zip(axes[1:], factors):
    sampled = sample_image(demo_input, factor=factor)
    ax.imshow(sampled[0].permute(1, 2, 0).cpu())
    ax.set_title(f"Sampling factor {factor}")
for ax in axes:
    ax.axis("off")
fig.savefig(OUTPUT / "demo_sample_image.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
checks.check_sampling(sample_image)

### B. Intensity quantization
Quantization represents pixel intensities using a finite number of levels.
Implement $Q_b(x)=\mathrm{round}[(2^b-1)x]/(2^b-1)$ for integer $1\leq b\leq8$.
Eight bits preserves the original 8-bit samples. Lower bit depths retain fewer intensity levels.

In [ ]:
def quantize(images: torch.Tensor, bits: int) -> torch.Tensor:
    """Reduce intensity resolution to 2**bits uniformly spaced levels in [0, 1].

    Args:
        images: Float RGB batch in [0, 1], shaped (B, 3, H, W).
            B is the number of images; H and W are height and width.
        bits: Integer bit depth from 1 to 8, applied to each color channel.

    Returns:
        Quantized batch with the same shape, dtype, and device as images.
        The input is not modified."""
    raise NotImplementedError("Implement quantize to test this operator.")

In [ ]:
demo_input = case_images[1:2].clone()
bit_depths = [6, 4, 2, 1]
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5), layout="constrained")
axes[0].imshow(demo_input[0].permute(1, 2, 0).cpu())
axes[0].set_title("Input image")
for ax, bits in zip(axes[1:], bit_depths):
    quantized = quantize(demo_input, bits=bits)
    ax.imshow(quantized[0].permute(1, 2, 0).cpu())
    ax.set_title(f"{bits} bit(s): {2**bits} levels/channel")
for ax in axes:
    ax.axis("off")
fig.savefig(OUTPUT / "demo_quantize.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
checks.check_quantize(quantize)

### C. Affine geometric transformation
Rotate counterclockwise by $\theta$ around $((W-1)/2, (H-1)/2)$ using inverse mapping and
bilinear interpolation. For centered coordinates $(x, y)$, compute
$x_s = x\cos\theta - y\sin\theta$ and $y_s = x\sin\theta + y\cos\theta$, then add the center back.
Fill outside regions beyond the source bounds with neutral gray (`0.5`) and clamp outputs
to [0, 1]. Zero angle returns a clone.

In [ ]:
def affine_rotation(images: torch.Tensor, angle: float) -> torch.Tensor:
    """Rotate around the center using inverse mapping and bilinear interpolation.

    Args:
        images: Float RGB batch in [0, 1], shaped (B, 3, H, W).
            B is the number of images; H and W are height and width.
        angle: Counterclockwise rotation in degrees, applied to every image
            about center ((W - 1) / 2, (H - 1) / 2). Zero returns a clone.

    Returns:
        Rotated batch with the same shape, dtype, and device as images,
        clamped (restricted) to [0, 1]. Regions outside the input image
        boundary must be filled with neutral gray (0.5). The input is not
        modified."""
    raise NotImplementedError("Implement affine_rotation to test this operator.")

In [ ]:
demo_input = case_images[1:2].clone()
angles = [-45, -20, 20, 45]
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5), layout="constrained")
axes[0].imshow(demo_input[0].permute(1, 2, 0).cpu())
axes[0].set_title("Input image")
for ax, angle in zip(axes[1:], angles):
    rotated = affine_rotation(demo_input, angle=angle)
    ax.imshow(rotated[0].permute(1, 2, 0).cpu())
    ax.set_title(f"Rotation {angle:+d} degrees")
for ax in axes:
    ax.axis("off")
fig.savefig(OUTPUT / "demo_affine_rotation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
checks.check_affine(affine_rotation)

### D1. Image histogram
A histogram counts how many pixels have each intensity level.
First convert RGB to grayscale using the channel mean, then round intensity*255 into 256 bins.
Return a separate row of counts for each image of shape `(B, 256)` on the input device.
Use this function to plot the histograms before and after equalization.

In [ ]:
def image_histogram(images: torch.Tensor) -> torch.Tensor:
    """Count pixels at each of 256 intensity levels, separately for every image.

    Args:
        images: Float RGB batch in [0, 1], shaped (B, 3, H, W).
            B is the number of images; H and W are height and width.

    Returns:
        Integer pixel counts shaped (B, 256), on the input device.
        Use the RGB channel mean as intensity and round intensity*255
        to select bins 0 through 255. Each histogram sums to H*W.
        The input is not modified."""
    raise NotImplementedError("Implement image_histogram to test this operator.")

In [ ]:
histogram_input = case_images[1:2].clone()
grayscale_input = histogram_input.mean(dim=1, keepdim=True)
counts = image_histogram(histogram_input)[0]
fig, axes = plt.subplots(1, 3, figsize=(12, 4), layout="constrained")
axes[0].imshow(histogram_input[0].permute(1, 2, 0).cpu())
axes[0].set_title("Input image")
axes[0].axis("off")
axes[1].imshow(grayscale_input[0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Grayscale intensity (RGB mean)")
axes[1].axis("off")
axes[2].bar(range(256), counts.cpu().numpy(), width=1)
axes[2].set(
    title="Input intensity histogram", xlabel="Intensity level", ylabel="Pixel count", xlim=(0, 255)
)
fig.savefig(OUTPUT / "demo_image_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
checks.check_image_histogram(image_histogram)

### D2. Histogram equalization
Histogram equalization redistributes intensity levels to spread contrast across their range.
Convert RGB to intensity using its channel mean. For each image, compute a 256-bin
histogram and cumulative distribution. Map intensities using (CDF−CDF_min)/(1−CDF_min).
You may call `image_histogram` to compute the per-image counts.
Leave constant-intensity images unchanged. Replicate the result into three channels.

In [ ]:
def histogram_equalize(images: torch.Tensor) -> torch.Tensor:
    """Redistribute grayscale intensities using each image's cumulative histogram.

    Args:
        images: Float RGB batch in [0, 1], shaped (B, 3, H, W).
            B is the number of images; H and W are height and width.

    Returns:
        Equalized grayscale batch in [0, 1], with intensity replicated across
        three channels. Shape, dtype, and device match images. Each image is
        equalized independently, and the input is not modified."""
    raise NotImplementedError("Implement histogram_equalize to test this operator.")

In [ ]:
demo_input = case_images[1:2].clone()
demo_output = histogram_equalize(demo_input)
fig, axes = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
for ax, image, title in zip(
    axes,
    [demo_input, demo_output],
    ["Input image", "Histogram-equalized grayscale"],
):
    ax.imshow(image[0].permute(1, 2, 0).cpu())
    ax.set_title(title)
    ax.axis("off")
fig.savefig(OUTPUT / "demo_histogram_equalize.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
input_counts = image_histogram(demo_input)[0]
output_counts = image_histogram(demo_output)[0]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharex=True, sharey=True, layout="constrained")
for ax, counts, title in zip(
    axes,
    [input_counts, output_counts],
    ["Input intensity histogram", "Equalized output histogram"],
):
    ax.bar(range(256), counts.cpu().numpy(), width=1)
    ax.set(title=title, xlabel="Intensity level", xlim=(0, 255))
axes[0].set_ylabel("Pixel count")
fig.savefig(OUTPUT / "histogram_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
checks.check_histogram(histogram_equalize)

### E. Gaussian smoothing
A smoothing filter replaces each pixel with a weighted neighborhood average.
Construct $e^{-(x^2+y^2)/(2\sigma^2)}$ within radius $\lceil3\sigma\rceil$ and
normalize its sum to one. Pad borders by the kernel radius using reflection padding
(mirroring border pixels), filter each RGB channel independently, and clamp to [0, 1]. Zero sigma returns a clone.

In [ ]:
def gaussian_smooth(images: torch.Tensor, sigma: float) -> torch.Tensor:
    """Smooth spatial detail using a normalized Gaussian-weighted neighborhood.

    Args:
        images: Float RGB batch in [0, 1], shaped (B, 3, H, W).
            B is the number of images; H and W are height and width.
        sigma: Nonnegative Gaussian standard deviation in pixels; zero is identity.
            Use kernel radius ceil(3*sigma), smaller than H and W.

    Returns:
        Smoothed batch with the same shape, dtype, and device as images.
        Filter RGB channels independently using reflection border padding
        (mirroring interior edge pixels) of width ceil(3*sigma), and clamp
        outputs to [0, 1]. The input is not modified."""
    raise NotImplementedError("Implement gaussian_smooth to test this operator.")

In [ ]:
demo_input = case_images[1:2].clone()
sigmas = [0.5, 1, 2, 4]
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5), layout="constrained")
axes[0].imshow(demo_input[0].permute(1, 2, 0).cpu())
axes[0].set_title("Input image")
for ax, sigma in zip(axes[1:], sigmas):
    smoothed = gaussian_smooth(demo_input, sigma=sigma)
    ax.imshow(smoothed[0].permute(1, 2, 0).cpu())
    ax.set_title(f"Gaussian sigma {sigma:g}")
for ax in axes:
    ax.axis("off")
fig.savefig(OUTPUT / "demo_gaussian_smooth.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
checks.check_gaussian_smooth(gaussian_smooth)

### F. Laplacian sharpening
The Laplacian measures second-order intensity changes. Filter each RGB channel
independently with the mask [[0,1,0],[1,-4,1],[0,1,0]] using reflection border padding.
Sharpen with $g=f-a\nabla^2f$, for $a\geq0$, then clip to [0,1].
Keep the signed response until the final subtraction; do not take its absolute value.

In [ ]:
def laplacian_sharpen(images: torch.Tensor, amount: float) -> torch.Tensor:
    """Emphasize intensity changes by subtracting a scaled Laplacian response.

    Args:
        images: Float RGB batch in [0, 1], shaped (B, 3, H, W).
            B is the number of images; H and W are height and width.
        amount: Nonnegative sharpening strength; zero is identity.

    Returns:
        Sharpened batch with the same shape, dtype, and device as images.
        Filter RGB channels independently with the four-neighbor Laplacian
        (center -4) using reflection border padding (mirroring interior edge
        pixels), subtract amount * response, and clip (clamp) final values
        to [0, 1]. The input is not modified."""
    raise NotImplementedError("Implement laplacian_sharpen to test this operator.")

In [ ]:
demo_input = case_images[1:2].clone()
amounts = [0.25, 0.5, 1, 2]
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5), layout="constrained")
axes[0].imshow(demo_input[0].permute(1, 2, 0).cpu())
axes[0].set_title("Input image")
for ax, amount in zip(axes[1:], amounts):
    sharpened = laplacian_sharpen(demo_input, amount=amount)
    ax.imshow(sharpened[0].permute(1, 2, 0).cpu())
    ax.set_title(f"Sharpening amount {amount:g}")
for ax in axes:
    ax.axis("off")
fig.savefig(OUTPUT / "demo_laplacian_sharpen.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
checks.check_laplacian_sharpen(laplacian_sharpen)

## 4. Compare your results
The evaluation below calls the six operator functions you just wrote.
The grayscale comparison isolates histogram equalization from the removal of color.
Rotation can also crop objects; inspect the images before interpreting accuracy changes.

**Before running:** Based on your visual demos above, which operations and parameter levels should change predictions most? Why?

**Your prediction:** TODO

In [ ]:
reload(experiments)  # pick up updated evaluation conditions in an existing kernel
conditions = experiments.lab_conditions(
    sample_image,
    quantize,
    affine_rotation,
    histogram_equalize,
    gaussian_smooth,
    laplacian_sharpen,
)
example_indices = [1, 6]
selected = [
    c
    for c in conditions
    if c.key
    in {
        "clean",
        "sampling_8",
        "quantization_2",
        "quantization_1",
        "rotation_-45",
        "rotation_45",
        "grayscale",
        "equalized",
    }
]
fig = transformation_gallery(
    model,
    case_images[example_indices],
    case_labels[example_indices],
    case_ids[example_indices],
    selected,
)
fig.savefig(OUTPUT / "examples.png", dpi=150, bbox_inches="tight")
plt.show()
filter_examples = [
    c
    for c in conditions
    if c.key in {"clean", "gaussian_2", "gaussian_4", "laplacian_1", "laplacian_2"}
]
fig = transformation_gallery(
    model,
    case_images[example_indices],
    case_labels[example_indices],
    case_ids[example_indices],
    filter_examples,
)
fig.savefig(OUTPUT / "filter_examples.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
table, results = run_suite(
    model,
    loader,
    conditions,
    OUTPUT,
    baseline=clean,
    metadata={"subset_manifest": "data/lab_subset.json"},
)
display(
    table[
        [
            "condition",
            "accuracy_pct",
            "drop_pp",
            "flip_pct",
            "confidence_mean",
            "image_mse_mean",
        ]
    ].round(3)
)
fig = plot_curves(table)
fig.savefig(OUTPUT / "accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

Accuracy drop is measured in percentage points from the original images.
Prediction flips can introduce errors or correct them.

**Discuss:** Did your results match your prediction? Explain the effects of sampling,
intensity resolution, interpolation, and contrast. Did histogram equalization help
compared with grayscale? How do smoothing and sharpening affect detail and predictions?
Support your answers with numbers and examples.

**Your discussion:** TODO